<a href="https://colab.research.google.com/github/iqrahussainn/Crime-investigation-system/blob/main/krr_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crime Investigation System

| **Topic Name**               | **What **                               | ** project**                            |
| ---------------------------- | ----------------------------------------------------------- | ---------------------------------------------- |
| **Knowledge Representation** | Writing crime information in a way the computer understands | Weapon, motive, violence are written as inputs |
| **Rule-Based System**        | Using fixed rules to decide something                       | Rules decide the crime                         |
| **IF–THEN Rules**            | “If this happens, then do that”                             | Gun + violence → murder                        |
| **Forward Chaining**         | Start from input and move step by step to result            | Input → rules → crime                          |
| **Inference Engine**         | The “brain” that checks rules                               | Experta checks the rules                       |
| **Conflict Resolution**      | Choosing one answer if many appear                          | Picks the final crime                          |
| **Default Rule**             | Gives an answer even if rules fail                          | System never stays empty                       |
| **Explainable Reasoning**    | Telling why the answer came                                 | Shows rule explanation                         |
| **System Structure**         | Separating thinking and screen                              | Rules + Gradio UI                              |


# — Setup cell

In [ ]:
# prepares the environment and installs required libraries.
# Python 3.12 compatibility patch for experta
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping
collections.Sequence = collections.abc.Sequence

!pip install experta gradio pandas


# — Create CSV dataset

In [ ]:
#  manually creates crime investigation data.
import pandas as pd

data = [
    ["C001","Knife","Revenge","No","Yes","No","Yes","Home","Night","Yes","Murder","Person A"],
    ["C002","None","Greed","Yes","No","Yes","No","Shop","Day","No","Burglary","Person B"],
    ["C003","Gun","Anger","No","Yes","No","Yes","Street","Night","Yes","Murder","Person C"],
    ["C004","Blunt","Objective","Yes","Yes","No","Yes","Office","Day","Yes","Assault","Person D"],
    ["C005","None","Opportunity","No","No","Yes","No","Home","Night","No","Theft","Unknown"],
    ["C006","Knife","Jealousy","Yes","Yes","No","Yes","Home","Evening","Yes","Attempted Murder","Person E"],
]

columns = [
    "Case_ID","Weapon","Motive","Witness","Fingerprint",
    "Theft","Violence","Location","Time",
    "Suspect_Nearby","Crime_Type","Primary_Suspect"
]

df = pd.DataFrame(data, columns=columns)
df.to_csv("crime_investigation_dataset.csv", index=False)

df


,Case_ID,Weapon,Motive,Witness,Fingerprint,Theft,Violence,Location,Time,Suspect_Nearby,Crime_Type,Primary_Suspect
0,C001,Knife,Revenge,No,Yes,No,Yes,Home,Night,Yes,Murder,Person A
1,C002,None,Greed,Yes,No,Yes,No,Shop,Day,No,Burglary,Person B
2,C003,Gun,Anger,No,Yes,No,Yes,Street,Night,Yes,Murder,Person C
3,C004,Blunt,Objective,Yes,Yes,No,Yes,Office,Day,Yes,Assault,Person D
4,C005,None,Opportunity,No,No,Yes,No,Home,Night,No,Theft,Unknown
5,C006,Knife,Jealousy,Yes,Yes,No,Yes,Home,Evening,Yes,Attempted Murder,Person E


# — Load dataset

In [ ]:
#  loads the dataset for reasoning.
df = pd.read_csv("crime_investigation_dataset.csv")
df.head()


,Case_ID,Weapon,Motive,Witness,Fingerprint,Theft,Violence,Location,Time,Suspect_Nearby,Crime_Type,Primary_Suspect
0,C001,Knife,Revenge,No,Yes,No,Yes,Home,Night,Yes,Murder,Person A
1,C002,NaN,Greed,Yes,No,Yes,No,Shop,Day,No,Burglary,Person B
2,C003,Gun,Anger,No,Yes,No,Yes,Street,Night,Yes,Murder,Person C
3,C004,Blunt,Objective,Yes,Yes,No,Yes,Office,Day,Yes,Assault,Person D
4,C005,NaN,Opportunity,No,No,Yes,No,Home,Night,No,Theft,Unknown


# — CrimeCase (Fact class)

| **Fact Name**  | **Description **                | **Possible Values**                                             |
| -------------- | ---------------------------------------------------- | --------------------------------------------------------------- |
| Weapon         | Type of weapon used in the crime                     | Gun, Knife, Blunt, None                                         |
| Motive         | Reason behind committing the crime                   | Revenge, Greed, Anger, Opportunity, Jealousy, Personal Conflict |
| Witness        | Indicates presence of an eyewitness                  | Yes, No                                                         |
| Fingerprint    | Indicates availability of fingerprint evidence       | Yes, No                                                         |
| Theft          | Indicates whether theft was involved                 | Yes, No                                                         |
| Violence       | Indicates whether violence occurred                  | Yes, No                                                         |
| Location       | Place where the crime occurred                       | Home, Shop, Street, Office, Public                              |
| Time           | Time of occurrence of the crime                      | Day, Evening, Night                                             |
| Suspect_Nearby | Indicates whether a suspect was near the crime scene | Yes, No                                                         |


In [ ]:
# Import Fact class from experta
# Fact is used to store information (knowledge) for the expert system
from experta import Fact

# CrimeCase class inherits from Fact
# This class represents all known facts about a crime case

class CrimeCase(Fact):
    """
    Represents investigation facts for a crime case
    """
    pass


# — Expert System + Rules

| **Rule No.**      | **IF Conditions (Facts)**                               | **THEN Conclusion (Crime Type)** | **Explanation**                                |
| ----------------- | ------------------------------------------------------- | -------------------------------- | ------------------------------------------------------------- |
| R1                | Weapon = Gun AND Violence = Yes                         | Murder                           | Use of a gun with violence strongly indicates murder.         |
| R2                | Weapon = Knife AND Motive = Revenge                     | Murder                           | Revenge motive with a knife suggests intentional killing.     |
| R3                | Weapon = Knife AND Violence = Yes AND Fingerprint = Yes | Attempted Murder                 | Violent knife attack with evidence suggests attempted murder. |
| R4                | Theft = Yes AND Location = Shop                         | Burglary                         | Theft occurring in a shop indicates burglary.                 |
| R5                | Theft = Yes AND Violence = No                           | Theft                            | Theft without violence is classified as theft.                |
| R6                | Violence = Yes AND Weapon = Blunt                       | Assault                          | Violence using a blunt object indicates assault.              |
| R7 (Default Rule) | No specific rule matches                                | Theft (default)                  | Default rule ensures the system always gives an output.       |


In [ ]:
# Import required classes from experta
# KnowledgeEngine → main engine for rule-based reasoning
# Rule → used to define IF–THEN rules
from experta import KnowledgeEngine, Rule

class CrimeExpertSystem(KnowledgeEngine):

    def __init__(self):
        super().__init__()
        self.predicted_crimes = []
        self.explanations = []

    # --- Murder ---
    @Rule(CrimeCase(Weapon="Gun", Violence="Yes"))
    def gun_murder(self):
        self.predicted_crimes.append("Murder")
        self.explanations.append("Gun used with violence indicates murder.")

    @Rule(CrimeCase(Weapon="Knife", Motive="Revenge"))
    def knife_revenge_murder(self):
        self.predicted_crimes.append("Murder")
        self.explanations.append("Revenge motive with knife suggests murder.")

    # --- Attempted Murder ---
    @Rule(CrimeCase(Weapon="Knife", Violence="Yes", Fingerprint="Yes"))
    def attempted_murder(self):
        self.predicted_crimes.append("Attempted Murder")
        self.explanations.append("Violent knife attack with evidence suggests attempted murder.")

    # --- Burglary ---
    @Rule(CrimeCase(Theft="Yes", Location="Shop"))
    def burglary(self):
        self.predicted_crimes.append("Burglary")
        self.explanations.append("Theft at shop indicates burglary.")

    # --- Theft ---
    @Rule(CrimeCase(Theft="Yes", Violence="No"))
    def theft(self):
        self.predicted_crimes.append("Theft")
        self.explanations.append("Theft without violence indicates theft.")

    # --- Assault ---
    @Rule(CrimeCase(Violence="Yes", Weapon="Blunt"))
    def assault(self):
        self.predicted_crimes.append("Assault")
        self.explanations.append("Blunt weapon violence suggests assault.")

    # --- Fallback Rule ---
    @Rule(CrimeCase())
    def default_rule(self):
        if not self.predicted_crimes:
            self.predicted_crimes.append("Theft")
            self.explanations.append("Default rule applied due to insufficient evidence.")


Law and punishment


| **Crime Type**       | **Legal Reference ** | **Punishment / Description**             |
| -------------------- | ----------------------------- | ---------------------------------------- |
| **Murder**           | Section 302                   | Punishable by death or life imprisonment |
| **Attempted Murder** | Section 324                   | Imprisonment up to 10 years              |
| **Assault**          | Section 351                   | Fine or imprisonment                     |
| **Theft**            | Section 378                   | Imprisonment up to 3 years or fine       |
| **Burglary**         | Section 457                   | Imprisonment up to 14 years              |


In [ ]:
# --- Legal Knowledge Table ---
# This dictionary maps each type of crime to its corresponding legal section
# and the associated punishment according to law.
# Law table: maps crime type to legal section and punishment
legal_knowledge = {
    "Murder": {
        "section": "Section 302",
        "punishment": "Death or life imprisonment"
    },
    "Attempted Murder": {
        "section": "Section 324",
        "punishment": "Up to 10 years imprisonment"
    },
    "Assault": {
        "section": "Section 351",
        "punishment": "Fine or imprisonment"
    },
    "Theft": {
        "section": "Section 378",
        "punishment": "Up to 3 years imprisonment or fine"
    },
    "Burglary": {
        "section": "Section 457",
        "punishment": "Up to 14 years imprisonment"
    }
}

def get_punishment(crime_type):
    if crime_type in legal_knowledge:
        law = legal_knowledge[crime_type]
        return f"{law['section']} — {law['punishment']}"
    else:
        return "No legal information available"




# — Final prediction function

In [ ]:
# This function selects the final crime when multiple rules fire.
def final_prediction(predictions):
    return max(set(predictions), key=predictions.count)


# — Row to Fact

In [ ]:
# This function converts dataset rows into symbolic facts.
def row_to_fact(row):
    return CrimeCase(
        Weapon=row["Weapon"],
        Motive=row["Motive"],
        Witness=row["Witness"],
        Fingerprint=row["Fingerprint"],
        Theft=row["Theft"],
        Violence=row["Violence"],
        Location=row["Location"],
        Time=row["Time"],
        Suspect_Nearby=row["Suspect_Nearby"]
    )


# — Test the system

In [ ]:
# This cell tests the expert system on sample cases.
for i, row in df.iterrows():
    engine = CrimeExpertSystem()
    engine.reset()
    engine.declare(row_to_fact(row))
    engine.run()

    print("Case:", row["Case_ID"])
    print("Actual Crime:", row["Crime_Type"])
    print("Predicted Crime:", final_prediction(engine.predicted_crimes))
    print("Explanation:", engine.explanations)
    print("-" * 50)


Case: C001
Actual Crime: Murder
Predicted Crime: Murder
Explanation: ['Default rule applied due to insufficient evidence.', 'Revenge motive with knife suggests murder.', 'Violent knife attack with evidence suggests attempted murder.']
--------------------------------------------------
Case: C002
Actual Crime: Burglary
Predicted Crime: Theft
Explanation: ['Default rule applied due to insufficient evidence.', 'Theft at shop indicates burglary.', 'Theft without violence indicates theft.']
--------------------------------------------------
Case: C003
Actual Crime: Murder
Predicted Crime: Murder
Explanation: ['Default rule applied due to insufficient evidence.', 'Gun used with violence indicates murder.']
--------------------------------------------------
Case: C004
Actual Crime: Assault
Predicted Crime: Assault
Explanation: ['Default rule applied due to insufficient evidence.', 'Blunt weapon violence suggests assault.']
--------------------------------------------------
Case: C005
Actual C

# — Prediction function for Gradio

In [ ]:
# This cell creates the user interface for interaction.
def predict_crime(
    Weapon, Motive, Witness, Fingerprint,
    Theft, Violence, Location, Time, Suspect_Nearby
):
    # Create crime case from user input
    case = CrimeCase(
        Weapon=Weapon,
        Motive=Motive,
        Witness=Witness,
        Fingerprint=Fingerprint,
        Theft=Theft,
        Violence=Violence,
        Location=Location,
        Time=Time,
        Suspect_Nearby=Suspect_Nearby
    )

    # Initialize and run expert system
    engine = CrimeExpertSystem()
    engine.reset()
    engine.declare(case)
    engine.run()

    # Get results
    predicted_crime = final_prediction(engine.predicted_crimes)
    explanation = "\n".join(engine.explanations)
    punishment = get_punishment(predicted_crime)

    # Return results to Gradio
    return predicted_crime, explanation, punishment


# — Gradio UI

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("## Crime Investigation Expert System")
    gr.Markdown("Rule-based Knowledge Representation & Reasoning (KR&R)")

    # ================= INPUTS =================
    weapon_input = gr.Dropdown(
        ["Knife", "Gun", "Blunt", "None"],
        label="Weapon"
    )

    motive_input = gr.Dropdown(
        ["Revenge", "Greed", "Anger", "Opportunity", "Jealousy", "Objective", "Personal Conflict"],
        label="Motive"
    )

    witness_input = gr.Radio(
        ["Yes", "No"],
        label="Witness"
    )

    fingerprint_input = gr.Radio(
        ["Yes", "No"],
        label="Fingerprint"
    )

    theft_input = gr.Radio(
        ["Yes", "No"],
        label="Theft"
    )

    violence_input = gr.Radio(
        ["Yes", "No"],
        label="Violence"
    )

    location_input = gr.Dropdown(
        ["Home", "Shop", "Street", "Office", "Public"],
        label="Location"
    )

    time_input = gr.Dropdown(
        ["Day", "Evening", "Night"],
        label="Time"
    )

    suspect_input = gr.Radio(
        ["Yes", "No"],
        label="Suspect Nearby"
    )

    # ================= BUTTON =================
    submit_btn = gr.Button("Submit")

    # ================= OUTPUTS =================
    crime_output = gr.Textbox(label="Predicted Crime")
    explanation_output = gr.Textbox(label="Reasoning Explanation")
    punishment_output = gr.Textbox(label="Applicable Law & Punishment")

    # ================= CONNECT BUTTON =================
    submit_btn.click(
        fn=predict_crime,
        inputs=[
            weapon_input,
            motive_input,
            witness_input,
            fingerprint_input,
            theft_input,
            violence_input,
            location_input,
            time_input,
            suspect_input
        ],
        outputs=[
            crime_output,
            explanation_output,
            punishment_output
        ]
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37593d28dc4a385699.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
